<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/Emotion_Detection(IITB_Techconnect_proj_)4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
!pip install scikit-plot

In [ ]:
# Core
import os
import random
import warnings
warnings.simplefilter("ignore")

# Numerical & Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Deep Learning (USE ONLY tensorflow.keras)
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
    Input
)
from tensorflow.keras.optimizers import Adam, RMSprop, SGD, Adamax
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.regularizers import l1, l2
from tensorflow.keras.utils import to_categorical, plot_model


In [ ]:
from google.colab import files

# Upload the fer2013.csv file
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from glob import glob
import matplotlib.pyplot as plt
import seaborn as sns
import os,time, random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Subset,DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ananthu017/emotion-detection-fer")

print("Path to dataset files:", path)

In [ ]:
data_path='/kaggle/input/emotion-detection-fer'
data=os.listdir(data_path)
print(f"Total folders in this Dataset: {len(data)}")
print(f"Files in this Dataset: {data}")

In [ ]:
# Use the 'path' variable from the kagglehub download as the base directory
base_data_dir = path

# List the contents of the base directory to get the 'splits' (e.g., 'train', 'test')
splits = os.listdir(base_data_dir)

for split in splits:
    split_path = os.path.join(base_data_dir, split)
    # Ensure that 'split_path' is actually a directory before proceeding
    if os.path.isdir(split_path):
        classes = os.listdir(split_path)
        print(f"{split} classes:", classes)

        counts = {cls: len(os.listdir(os.path.join(split_path, cls))) for cls in classes}
        print(f"{split} samples per class:", counts)
    else:
        print(f"Skipping non-directory item in base_data_dir: {split}")

In [ ]:
# Visualizing sample images
# Use the correct base path 'path' for the dataset
train_dir = os.path.join(path, "train")

example_path = glob(os.path.join(path, "train", "*", "*.png"))
print("Total training images:", len(example_path))

# 7 Random Images (1 images from each Class)
fig, axes = plt.subplots(1, 7, figsize=(22,8))
for ax in axes:
    img_path = random.choice(example_path)
    label = img_path.split(os.sep)[-2]
    img = Image.open(img_path).convert("L")  # grayscale
    ax.imshow(img, cmap="gray")
    ax.set_title(label)
    ax.axis("off")
plt.show()
print("Image size (W x H):", img.size)

In [ ]:
#Transfor ->to tensor
train_tf = transforms.Compose([transforms.Grayscale(num_output_channels=1),
                            transforms.Resize((48,48)),
                            transforms.RandomHorizontalFlip(p=0.5),
                            transforms.RandomRotation(10),
                            transforms.ToTensor()])

eval_tf = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48,48)),
    transforms.ToTensor(),
])

# Train Dataset
train_dataset = datasets.ImageFolder(os.path.join(path, "train"), transform=None)
print(f"Train set size before augmentation and split : {len(train_dataset)}")

class_names = train_dataset.classes
y=np.array(train_dataset.targets)
idxs=np.arange(len(y))

#Stratified split ( %20 validation data)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, val_idx = next(splitter.split(idxs, y))

# Train and Validation Dataset
full_train_aug = datasets.ImageFolder(os.path.join(path, "train"), transform=train_tf)
full_train_eval = datasets.ImageFolder(os.path.join(path, "train"), transform=eval_tf)

train_dataset = Subset(full_train_aug, train_idx)
val_dataset   = Subset(full_train_eval, val_idx)

# Test Dataset
test_dataset  = datasets.ImageFolder(os.path.join(path, "test"),  transform=eval_tf)

print(f"Train set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")
print(f"Test set size: {len(test_dataset)}")

# Data Loaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

In [ ]:
class CNNEmotion(nn.Module):
    def __init__(self, num_classes=7):
        super(CNNEmotion, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # Input: 1 Channel  -> Output: 32 feature
            nn.BatchNorm2d(32), # Batch Normalization
            nn.ReLU(), # Activation
            nn.Dropout(0.25), # Dropout for regularization
            nn.MaxPool2d(2, 2),  # Downsample: 48×48 -> 24×24

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # Input: 32  -> Output: 64 feature
            nn.BatchNorm2d(64),
            nn.ReLU(), # Activation
            nn.Dropout(0.25), # Dropout for regularization
            nn.MaxPool2d(2, 2),  # Downsample: 24×24 -> 12×12

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # Input: 64  -> Output: 128 feature
            nn.BatchNorm2d(128),
            nn.ReLU(), # Activation
            nn.Dropout(0.25), # Dropout for regularization
            nn.MaxPool2d(2, 2),  # Downsample: 12×12 -> 6×6

            nn.Conv2d(128, 256, kernel_size=3, padding=1), # Input: 128  -> Output: 256 feature
            nn.BatchNorm2d(256),
            nn.ReLU(), # Activation
            nn.Dropout(0.25), # Dropout for regularization
            nn.MaxPool2d(2, 2)   # Downsample: 6×6 -> 3×3
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(256*3*3, 128),  # Flattened features (256*3*3) → Hidden layer (128)
            nn.BatchNorm1d(128), # BatchNorm for fully connected layer
            nn.ReLU(), # Activation
            nn.Dropout(0.5), # Dropout for regularization
            nn.Linear(128, num_classes) # Hidden (128) → Output classes
        )

    def forward(self, x):
        x = self.conv_layers(x)  # Feature extraction
        x = x.view(x.size(0), -1) # Flatten features
        x = self.fc_layers(x)  # Classification
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNEmotion(num_classes=len(class_names)).to(device)

In [ ]:
def accuracy_from_logits(logits, y): # Computing Accuracy

    preds = logits.argmax(dim=1)              # class with highest logit = predicted label
    return (preds == y).float().mean().item() # mean of correct predictions

def run_epoch(model, loader, optimizer=None):

    is_train = optimizer is not None
    model.train(is_train)   # switch between train/eval mode
    total_loss, total_acc, n = 0.0, 0.0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        # Forward pass
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        if is_train:
            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
            optimizer.step()

        # Track statistics
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_from_logits(logits, y) * bs
        n += bs

    return total_loss / n, total_acc / n

In [ ]:
EPOCHS = 20
LR = 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
best_val_loss = float('inf')   # best score so far
patience = 5                   # early stopping patience
wait = 0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

t0_all = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # Training epoch
    train_loss, train_acc = run_epoch(model, train_loader, optimizer=optimizer)

    # Validation epoch
    val_loss,   val_acc   = run_epoch(model, val_loader,   optimizer=None)

    # Save metrics
    history["train_loss"].append(train_loss); history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss);     history["val_acc"].append(val_acc)

    # Scheduler update (on validation loss)
    scheduler.step(val_loss)

    print(f"Epoch {epoch:02d} | "
          f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f} | "
          f"{time.time() - t0:.1f}s")

    # Early stopping + checkpoint
    if val_loss + 1e-6 < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(model.state_dict(), "best_cnn.pt")   # save best model
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered.")
            break

print(f"Total training time: {time.time() - t0_all:.1f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy Curves
axes[0].plot(history["train_acc"], label="train_acc")
axes[0].plot(history["val_acc"],   label="val_acc")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy Curves")
axes[0].legend()

#  Loss Curves
axes[1].plot(history["train_loss"], label="train_loss")
axes[1].plot(history["val_loss"],   label="val_loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss Curves")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_model = CNNEmotion(num_classes=len(class_names)).to(device)
best_model.load_state_dict(torch.load("best_cnn.pt", map_location=device))
best_model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        logits = best_model(x)
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy().tolist())
        all_true.extend(y.cpu().numpy().tolist())

test_acc = (np.array(all_preds) == np.array(all_true)).mean()

In [ ]:
print("Test accuracy:", test_acc)


In [ ]:
print("\nClassification report:\n")
print(classification_report(all_true, all_preds, target_names=class_names))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45, ha="right")
plt.yticks(tick_marks, class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
def show_random_test_predictions(model, dataset, n=8):
    idxs = random.sample(range(len(dataset)), n)
    imgs, labels = zip(*[dataset[i] for i in idxs])   # list of (C,H,W) tensors + int labels

    batch = torch.stack(imgs).to(device)              # [n,1,48,48]
    labels_t = torch.tensor(labels).to(device)

    with torch.no_grad():
        logits = model(batch)
        probs  = F.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)

    # plot
    rows, cols = 2, n // 2 if n % 2 == 0 else (n // 3 + 1)
    cols = n // rows + (n % rows > 0)
    plt.figure(figsize=(3*cols, 3*rows))
    for i in range(n):
        plt.subplot(rows, cols, i+1)
        img = imgs[i].squeeze(0).cpu().numpy()       # (48,48)
        plt.imshow(img, cmap="gray")
        true_label = classes[labels[i]]
        pred_label = classes[preds[i].item()]
        color = "green" if true_label == pred_label else "red"
        plt.title(f"P: {pred_label}\nT: {true_label}", color=color, fontsize=10)
        plt.axis("off")
    plt.suptitle("Random Test Images: Predictions vs Ground Truth", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
show_random_test_predictions(best_model, test_dataset, n=16)
